Section 1 : Sujet et problématique

Configuration

In [7]:
# F8 : Battre la marche aléatoire

**Nom et prénom :** MALAK SABRI   **Code Apogée :** 24010191   **Groupe :FIN 3   **Filière :** FINANCE   **Date :** 17/11/2026

**Dépôt GitHub :** https://github.com/votre-identifiant/24010191-SABRIMALAK-FIN3

## 1. Sujet et problématique
- **Intérêt du sujet** : si les rendements d'un indice boursier suivent une marche aléatoire, aucune règle fondée sur les cours passés ne peut battre le hasard (hypothèse d'efficience faible). Un gérant ou un étudiant en finance veut savoir si les rendements et la volatilité récents permettent de prévoir le signe du lendemain, et décider s'il vaut la peine de bâtir une stratégie dessus.
- **Problématique** : un modèle de classification utilisant uniquement les rendements passés du S&P 500 prévoit-il le signe du rendement du jour suivant mieux que la référence naïve, sur une période de test postérieure à l'apprentissage ?
- **Objectif mesurable (fixé avant de voir le test)** : métrique principale = accuracy sur le test (20 % des données les plus récentes). Réussite si l'accuracy dépasse d'au moins 2 points la meilleure des deux références (classe majoritaire de l'apprentissage ; signe de la veille) **et** si un test binomial unilatéral contre cette référence est significatif à 5 %. Métriques secondaires : F1 et AUC.
- **Nature du problème** : classification binaire sur série temporelle (cible = 1 si le rendement du jour suivant est strictement positif, 0 sinon) : découpage chronologique obligatoire.
- **Algorithme principal** : n° 8, régression logistique. Comparaison facultative : n° 12, forêt aléatoire.

SyntaxError: unmatched ')' (647862154.py, line 8)

2. Jeu de données

In [4]:
brut = pd.read_csv(CHEMIN)
print("Dimensions brutes :", brut.shape)
print(brut.dtypes)
print(brut.head())

NameError: name 'pd' is not defined

In [5]:
col_date = [c for c in brut.columns if c.lower() in ("observation_date", "date")][0]
col_val = [c for c in brut.columns if c != col_date][0]
df = brut.rename(columns={col_date: "date", col_val: "prix"})
df["date"] = pd.to_datetime(df["date"])
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
n_manq = int(df["prix"].isna().sum())
pct_manq = df["prix"].isna().mean() * 100
df = df.set_index("date").sort_index()
n_dup = int(df.index.duplicated().sum())
df = df.dropna()
print(f"Valeurs manquantes : {n_manq} ({pct_manq:.2f} %) | doublons de dates : {n_dup}")
print("Observations :", len(df), "| période :", df.index.min().date(), "->", df.index.max().date())

NameError: name 'brut' is not defined

In [ ]:
dire(f"""
- **Nom et source** : S&P 500 (cours de clôture quotidiens), Federal Reserve Bank of St. Louis (FRED), série `SP500`. Lien : https://fred.stlouisfed.org/series/SP500
- **Extraction** : {DATE_EXTRACTION}.
- **Licence** : données S&P Dow Jones Indices diffusées par FRED pour usage d'information ; usage ici strictement pédagogique (à confirmer sur la page « Notes » de la série).
- **Taille et fréquence** : {len(df)} observations quotidiennes (jours de bourse), du {df.index.min().date()} au {df.index.max().date()}.
- **Unité d'observation** : un jour de bourse.
- **Adéquation** : la question porte sur la prévisibilité du signe du rendement quotidien ; la série de clôtures suffit à construire la cible et les variables passées.
- **Limites** : environ {round((df.index.max()-df.index.min()).days/365.25)} ans de données seulement ; une seule série ; contient le krach de mars 2020 (régime de forte volatilité) ; prix sans dividendes ; aucun coût de transaction ; peu d'observations pour détecter un faible signal.
""")

dictionnaire des variables

In [ ]:
| Variable | Description | Type | Unité | Rôle | Manquants |
| --- | --- | --- | --- | --- | --- |
| prix | Cours de clôture du S&P 500 | numérique | points d'indice | brute | jours sans cotation retirés (section 3) |
| ret | Rendement logarithmique quotidien 100 × ln(P_t / P_{t-1}) | numérique | % | base des variables | 1re ligne retirée |
| ret_lag1 à ret_lag5 | ret décalé de 1 à 5 jours | numérique | % | explicative | aucun après nettoyage |
| moy5, moy20 | moyenne glissante de ret sur 5 / 20 jours se terminant en t | numérique | % | explicative | aucun après nettoyage |
| vol5, vol20 | écart-type glissant de ret sur 5 / 20 jours se terminant en t | numérique | % | explicative | aucun après nettoyage |
| y | 1 si le rendement de t+1 est > 0, sinon 0 | binaire | sans | **cible** | dernière ligne retirée |